In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

In [8]:
data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\Cancer Prediction v2. - DS1 (1).csv" 
df = pd.read_csv(data_path, delimiter=",")  

# Features and target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]


print(np.bincount(y))
smote = SMOTE(random_state=42, k_neighbors=5)
X, y = smote.fit_resample(X, y)
print(np.bincount(y))


# (7 : 1 : 2)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_cv, y_train, y_cv = train_test_split(X_temp, y_temp, test_size=1/8, random_state=42, stratify=y_temp)

#Before SMOTE
print(np.bincount(y_train), np.bincount(y_cv), np.bincount(y_test))
print(y_train.shape, y_cv.shape, y_test.shape)

[234  75]
[234 234]
[143 143] [20 21] [71 70]
(286,) (41,) (141,)


In [3]:
external_test_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\ET1 (1).csv"
df_external = pd.read_csv(external_test_path, delimiter=",")
X_external = df_external.iloc[:, :-1]
y_external = df_external.iloc[:, -1]

print(X_external)
print(y_external)

     200717_x_at  202192_s_at  203592_s_at  207574_s_at  209304_x_at  \
0       0.780951    -0.336587    -0.255506     0.550705     0.356276   
1      -0.351034    -1.208983    -0.688482    -2.158107    -2.155740   
2      -0.432727    -0.343613    -0.115302    -0.542080     0.240398   
3      -1.260738    -0.403174     0.496536    -1.772841    -1.350446   
4       0.809847     2.112079     0.310269     0.242549     0.279259   
..           ...          ...          ...          ...          ...   
240     1.578388     0.149185    -0.287055     2.202842     2.375767   
241    -0.107202     1.306598     0.238374     0.806519     0.838113   
242     1.846280    -0.008404    -0.524969    -0.592765    -0.387908   
243     0.791526     0.319873    -0.239817    -0.231235    -0.034546   
244    -0.060069     0.173836    -0.064435     0.363354     0.123174   

     212356_at  218686_s_at  219173_at  37796_at  
0    -1.011282     0.780860  -0.192744 -1.359188  
1    -0.252015    -0.978074   0.5

In [6]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Define the model
model = ExtraTreesClassifier(n_estimators=300, random_state=42)

# Feature subset you want to use
selected_features = [
    '200717_x_at', '202192_s_at', '203592_s_at', '207574_s_at',
    '209304_x_at', '212356_at', '218686_s_at', '219173_at', '37796_at'
]

# Train the model
model.fit(X_train[selected_features], y_train)

# Function to evaluate and print metrics
def evaluate_set(X, y, name):
    y_pred = model.predict(X[selected_features])
    y_proba = model.predict_proba(X[selected_features])[:, 1]  # probability for positive class
    acc = accuracy_score(y, y_pred)
    auc = roc_auc_score(y, y_proba)
    print(f"{name} Accuracy: {acc:.4f} | AUC: {auc:.4f}")
    return auc, acc

# Evaluate on all sets
evaluate_set(X_train, y_train, "Train")
evaluate_set(X_cv, y_cv, "CV")
evaluate_set(X_test, y_test, "Test")
evaluate_set(X_external, y_external, "External")

# Optional: Detailed report for Test set
print("\nClassification Report (Test set):")
print(classification_report(y_test, model.predict(X_test[selected_features])))


Train Accuracy: 1.0000 | AUC: 1.0000
CV Accuracy: 0.8293 | AUC: 0.9655
Test Accuracy: 0.8440 | AUC: 0.9483
External Accuracy: 0.9265 | AUC: 0.9973

Classification Report (Test set):
              precision    recall  f1-score   support

           0       0.90      0.77      0.83        71
           1       0.80      0.91      0.85        70

    accuracy                           0.84       141
   macro avg       0.85      0.84      0.84       141
weighted avg       0.85      0.84      0.84       141

